In [2]:
import pandas as pd

In [14]:
df = pd.read_csv ('../quantification/results/classification_results_2026-01-23_19-54.csv')

In [15]:
df['Dataset'] = df['Dataset'].str.replace('_', ' ').str.title()
df['Split'] = df['Split'].str.replace('_', ' ').str.title()

In [16]:
pivot = df.pivot_table(index=['Dataset', 'Split'],
                       columns='Model',
                       values='Macro F1',
                       aggfunc='first')
pivot = pivot[['MLP', 'GCN', 'SAGE']]

In [17]:
def bold_max(row):
    m = row.max()
    return [f"\\textbf{{{x:.4f}}}" if x == m else f"{x:.4f}" for x in row]

formatted_data = pivot.apply(bold_max, axis=1, result_type='expand')
formatted_data.columns = [f"\\multicolumn{{1}}{{c}}{{\\textbf{{{col}}}}}" for col in pivot.columns]

In [19]:
latex_code = formatted_data.to_latex(
    multirow=True,
    index=True,
    float_format="%.4f",
    column_format='ll|rrr',
    escape=False,
)
latex_code = latex_code.replace(r'\midrule', r'\hline')
latex_code = latex_code.replace(r'\bottomrule', r'\hline')

print(latex_code)

\begin{tabular}{ll|rrr}
\toprule
 &  & \multicolumn{1}{c}{\textbf{MLP}} & \multicolumn{1}{c}{\textbf{GCN}} & \multicolumn{1}{c}{\textbf{SAGE}} \\
Dataset & Split &  &  &  \\
\hline
\multirow[t]{6}{*}{Deezer Europe} & Split 0 & \textbf{0.5713} & 0.4678 & 0.5204 \\
 & Split 1 & \textbf{0.5695} & 0.4852 & 0.5651 \\
 & Split 2 & \textbf{0.6140} & 0.5487 & 0.6016 \\
 & Split 3 & 0.3471 & \textbf{0.5069} & 0.4619 \\
 & Split 4 & \textbf{0.5569} & 0.5051 & 0.5291 \\
 & Split 5 & \textbf{0.6165} & 0.5516 & 0.6062 \\
\cline{1-5}
\multirow[t]{3}{*}{Ogbn Arxiv} & Split 0 & 0.3040 & 0.3001 & \textbf{0.3125} \\
 & Split 1 & 0.3201 & 0.3781 & \textbf{0.3916} \\
 & Split 2 & 0.3121 & 0.3186 & \textbf{0.3459} \\
\cline{1-5}
\multirow[t]{4}{*}{Presidential Election} & Split 0 & 0.6099 & 0.6282 & \textbf{0.7763} \\
 & Split 1 & 0.8046 & 0.7159 & \textbf{0.8454} \\
 & Split 2 & 0.8161 & 0.7060 & \textbf{0.8261} \\
 & Split 3 & 0.7583 & 0.7419 & \textbf{0.8703} \\
\cline{1-5}
\multirow[t]{5}{*}{Twitch Gam

In [20]:
df = pd.read_csv('../quantification/results/quantification_results.csv')

In [3]:
df = pd.read_csv('../quantification/results/ogbn_quantify.csv', sep=';')

In [5]:
def final_table(dataframe):
    pivot_df = dataframe.pivot_table(
        index=['Dataset', 'Split', 'Classifier'],
        columns=['Method'],
        values=['MAE', 'KL']
    )
    pivot_df.columns = pivot_df.columns.swaplevel(0, 1)
    order = ['CC', 'ACC', 'PCC', 'PACC', 'KDEy', 'SIS-ACC', 'SIS-PACC']
    methods = [m for m in order if m in pivot_df.columns.levels[0]]
    pivot_df = pivot_df.reindex(columns=methods, level=0)
    pivot_df = pivot_df.reindex(columns=['MAE', 'KL'], level=1)

    classifier_order = ['MLP', 'GCN', 'SAGE']
    classifiers = [c for c in classifier_order if c in pivot_df.index.levels[2]]
    pivot_df = pivot_df.reindex(classifiers, level='Classifier')
    formatted_df = pivot_df.copy().astype(str)

    for label, group_indices in pivot_df.groupby(level=['Dataset', 'Split']).groups.items():
            for col in pivot_df.columns:
                values = pivot_df.loc[group_indices, col]
                min_val = values.min()

                for idx in group_indices:
                    val = pivot_df.loc[idx, col]
                    f_val = f"{val:.4f}"

                    if val == min_val:
                        formatted_df.loc[idx, col] = f"\\textbf{{{f_val}}}"
                    else:
                        formatted_df.loc[idx, col] = f_val

    methods = formatted_df.columns.levels[0]
    num_methods = len(methods)
    column_def = "ll"
    header_cells = []

    for i, m in enumerate(methods):
        is_last = (i == num_methods - 1)
        column_def += "|cc"
        header_align = "c" if is_last else "c|"
        header_cells.append(f"\\multicolumn{{2}}{{{header_align}}}{{\\textbf{{{m}}}}}")

    header_row_1 = " & & " + " & ".join(header_cells) + r" \\"

    metrics_cells = ["MAE & KL"] * num_methods
    header_row_2 = r"\textbf{Split} & \textbf{Classifier} & " + " & ".join(metrics_cells) + r" \\"
    latex_body_lines = []
    total_cols = 2 + (num_methods * 2)

    for dataset_name, group in formatted_df.groupby(level=0, sort=False):
        if latex_body_lines:
            latex_body_lines.append(r"\midrule")

        section_header = f"\\multicolumn{{{total_cols}}}{{l}}{{\\textbf{{{dataset_name}}}}} \\\\"
        latex_body_lines.append(section_header)
        latex_body_lines.append(r"\midrule")

        sub_df = group.droplevel(0)
        chunk_latex = sub_df.to_latex(
            header=False,
            index=True,
            index_names=False,
            multirow=True,
            column_format=None,
            escape=False
        )

        lines = chunk_latex.splitlines()
        clean_lines = [l for l in lines if "tabular" not in l and "rule" not in l]
        latex_body_lines.extend(clean_lines)

    final_body = "\n".join(latex_body_lines)

    latex_code = f"""
\\begin{{table}}[h!]
\\centering
\\scriptsize
\\setlength{{\\tabcolsep}}{{3pt}}
\\renewcommand{{\\arraystretch}}{{1.1}}

\\resizebox{{\\textwidth}}{{!}}{{%
    \\begin{{tabular}}{{{column_def}}}
    \\toprule
    {header_row_1}
    \\midrule
    {header_row_2}
    \\midrule
{final_body}
    \\bottomrule
    \\end{{tabular}}%
}}
\\caption{{Quantification results (mean squarred error and KL divergence).}}
\\end{{table}}
"""
    return latex_code

print(final_table(df))


\begin{table}[h!]
\centering
\scriptsize
\setlength{\tabcolsep}{3pt}
\renewcommand{\arraystretch}{1.1}

\resizebox{\textwidth}{!}{%
    \begin{tabular}{ll|cc|cc|cc|cc|cc|cc|cc}
    \toprule
     & & \multicolumn{2}{c|}{\textbf{CC}} & \multicolumn{2}{c|}{\textbf{ACC}} & \multicolumn{2}{c|}{\textbf{PCC}} & \multicolumn{2}{c|}{\textbf{PACC}} & \multicolumn{2}{c|}{\textbf{KDEy}} & \multicolumn{2}{c|}{\textbf{SIS-ACC}} & \multicolumn{2}{c}{\textbf{SIS-PACC}} \\
    \midrule
    \textbf{Split} & \textbf{Classifier} & MAE & KL & MAE & KL & MAE & KL & MAE & KL & MAE & KL & MAE & KL & MAE & KL \\
    \midrule
\multicolumn{16}{l}{\textbf{deezer_europe}} \\
\midrule
\multirow[t]{3}{*}{split_0} & MLP & \textbf{0.0227} & \textbf{0.0012} & \textbf{0.3028} & 2.8732 & \textbf{0.0894} & \textbf{0.0174} & \textbf{0.3028} & 2.8732 & \textbf{0.3028} & \textbf{2.8732} & 0.3028 & 2.8732 & 0.3028 & 2.8732 \\
 & GCN & 0.3029 & 0.1875 & 0.5826 & \textbf{0.9339} & 0.2176 & 0.0969 & 0.4953 & \textbf{0.5706} & 0